# N01 — Optical fields, units and sampling

**Week 1 · Beginner · 35–50 minutes · CPU / no credentials**

**Virtual apparatus:** a 632.8 nm low-power Gaussian laser, camera/beam profiler and aperture. **Observable:** irradiance in W/m², integrated power in W, and 1/e²-intensity waist in metres.

**Prerequisites:** Python/NumPy arrays, complex amplitudes and Gaussian integrals.

**Learning outcomes:** distinguish a field amplitude from irradiance; recover Gaussian beam power and waist; diagnose why source normalization can conceal window clipping.

PhotonicsLab represents a transverse, monochromatic Jones envelope with shape `(2,ny,nx)` in **√W/m**, not electric field in V/m. The grid is an FFT window in metres. The integrated square norm is power in watts in the specified paraxial model.

### Predict *before* you run
For an untruncated circular TEM00 beam with waist `w0` and power `P`:

`I(r) = [2P/(π w0²)] exp(-2r²/w0²)`, and `∫I dx dy = P`.

How will a window narrower than the beam change the inferred waist, **even when** software normalizes the sampled power to exactly P?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from photonicslab import Grid, GaussianBeam, CircularAperture

wavelength = 632.8e-9
w0 = 0.35e-3
power_w = 1e-3

def measured_waist_x(field):
    weights = field.intensity
    mean_x = np.sum(field.grid.mesh[0] * weights) / np.sum(weights)
    variance_x = np.sum((field.grid.mesh[0] - mean_x)**2 * weights) / np.sum(weights)
    return 2 * np.sqrt(variance_x)  # Gaussian 1/e² intensity radius

grid = Grid(nx=256, ny=256, width=4e-3, height=4e-3)
beam = GaussianBeam(wavelength=wavelength, waist=w0, power=power_w).emit(grid)
x, y = grid.mesh
analytic_irradiance = 2 * power_w / (np.pi * w0**2) * np.exp(-2*(x*x+y*y)/w0**2)

assert beam.amplitude.shape == (2, grid.ny, grid.nx)
assert np.isclose(beam.power, power_w, rtol=1e-12)
assert np.isclose(measured_waist_x(beam), w0, rtol=1e-3)
assert np.max(np.abs(beam.intensity - analytic_irradiance)) / analytic_irradiance.max() < 1e-3
print(f"Power: {beam.power*1e3:.6f} mW; waist: {measured_waist_x(beam)*1e6:.2f} µm; grid step: {grid.dx*1e6:.2f} µm")


### Inspect the beam and interrogate the window

The source is discretely normalized by PhotonicsLab; a correct reported *power* is **not** proof that the simulated spatial distribution fits the window. Compare three transverse widths at fixed physical beam waist. The `edge_power_fraction()` diagnostic measures the sampled outer border; it is not a universal bound on physical truncation error.


In [ ]:
windows = [(1.0e-3, 128), (2.0e-3, 192), (4.0e-3, 256)]
results = []
for extent, n in windows:
    b = GaussianBeam(wavelength=wavelength, waist=w0, power=power_w).emit(
        Grid(nx=n, ny=n, width=extent, height=extent))
    results.append((extent, b.power, measured_waist_x(b), b.edge_power_fraction()))
    print(f"window={extent*1e3:4.1f} mm  power={b.power*1e3:.4f} mW  "
          f"measured w={measured_waist_x(b)*1e6:.2f} µm  edge={b.edge_power_fraction():.4g}")
assert all(np.isclose(row[1], power_w, rtol=1e-12) for row in results)
assert results[0][3] > results[-1][3]
assert abs(results[-1][2] - w0) / w0 < 1e-3

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(beam.intensity, origin="lower", extent=[-2,2,-2,2])
fig.colorbar(im, ax=ax, label="Irradiance (W/m²)")
ax.set(xlabel="x (mm)", ylabel="y (mm)", title="Simulated Gaussian beam camera image")
plt.show()


### Aperture: compare a real prediction with a sampled integral

An ideal centred circular aperture of radius `a` has analytical transmitted fraction `T=1-exp(-2a²/w0²)` for an untruncated Gaussian. Pixelated aperture edges introduce a discrete error; change the grid resolution and verify convergence rather than tightening a tolerance blindly.


In [ ]:
radius = w0
fraction = CircularAperture(radius).apply(beam).power / beam.power
prediction = 1 - np.exp(-2 * (radius / w0)**2)
print(f"Aperture fraction (discrete): {fraction:.6f}; analytical: {prediction:.6f}")
assert abs(fraction - prediction) < 0.012


### Real-bench translation and numerical failure exercise

**Apparatus:** low-power, correctly classified laser source; approved beam profiler or calibrated camera; optical mounts; neutral density filters; irises; absorptive beam dump; optional apertures. Begin with your institution's laser safety procedure. No physical beam alignment procedure should encourage looking into a laser or making unapproved adjustments.

**Measure:** calibrate pixels to metres, subtract dark frame, avoid sensor saturation, account for detector gain and finite aperture, fit `I(r)` and integrate the measured profile. A sensor gives counts, **not** calibrated W/m² without calibration.

**Failure task:** repeat using a 0.6 mm window. Is its sampled power still 1 mW? Is the fitted waist reliable? Demonstrate that increasing window size *and* samples improves confidence.

**Record:** wavelength (vacuum), power, waist definition, `nx`, `ny`, extent, convergence results, uncertainty, source version and a plot.

**Limitations:** ideal paraxial coherent TEM00 at a waist; no lens aberrations, longitudinal fields, polarization mixing, sensor noise or physical laser safety simulation.

**Read:** Saleh & Teich, *Fundamentals of Photonics*, Gaussian beams; Goodman, *Introduction to Fourier Optics*, Fourier optics and sampling. **Next:** N02 free-space propagation.
